In [56]:
import pandas as pd
import matplotlib.pyplot as plt
import re

pd.options.future.infer_string = False

df = pd.read_csv('../data/raw/listings.csv')
print('Shape:', df.shape)
df.head()

Shape: (6120, 9)


,listing_id,property_id,agent_id,list_price,list_date,expiry_date,listing_status,days_on_market,description
0,LST-0001,PRP-3836,AGT-0264,434000.0,2021-01-28,2021-06-08,Sold,110.0,Building major hold buy western low chair.
1,LST-0002,PRP-0412,AGT-0041,1880600.0,2022-11-29,2023-03-02,Expired,145.0,Hope gas difficult think southern.
2,LST-0003,PRP-2661,AGT-0048,774800.0,2022-07-29,2023-01-11,Active,349.0,Much security purpose return image indeed nati...
3,LST-0004,PRP-3949,AGT-0148,788600.0,2021-04-23,2021-08-21,Expired,365.0,Heavy as agree beyond tax remember guy.
4,LST-0005,PRP-0655,AGT-0103,537700.0,2020-03-13,2020-08-23,Sold,NaN,Store own employee nearly throw morning. Reaso...


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6120 entries, 0 to 6119
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   listing_id      6120 non-null   object 
 1   property_id     6120 non-null   object 
 2   agent_id        6120 non-null   object 
 3   list_price      6120 non-null   float64
 4   list_date       6120 non-null   object 
 5   expiry_date     6120 non-null   object 
 6   listing_status  6120 non-null   object 
 7   days_on_market  5521 non-null   float64
 8   description     6120 non-null   object 
dtypes: float64(2), object(7)
memory usage: 430.4+ KB


In [58]:
df.isnull().sum()

listing_id          0
property_id         0
agent_id            0
list_price          0
list_date           0
expiry_date         0
listing_status      0
days_on_market    599
description         0
dtype: int64

In [59]:
df.duplicated().sum()

np.int64(120)

In [60]:
# Step 1: Remove Duplicate Rows

print("Duplicates found:", df.duplicated().sum())
print("Before:", df.shape[0], "rows")

df = df.drop_duplicates().reset_index(drop=True)

print("After: ", df.shape[0], "rows")

Duplicates found: 120
Before: 6120 rows
After:  6000 rows


In [61]:
# Step 2: Convert list_date and expiry_date to DateTime

print("Before:")
print("list_date dtype:  ", df['list_date'].dtype)
print("expiry_date dtype:", df['expiry_date'].dtype)
print("Sample list_date: ", df['list_date'].head(3).tolist())
print("Sample expiry_date:", df['expiry_date'].head(3).tolist())

df['list_date'] = pd.to_datetime(df['list_date'])
df['expiry_date'] = pd.to_datetime(df['expiry_date'])

print("\nAfter:")
print("list_date dtype:  ", df['list_date'].dtype)
print("expiry_date dtype:", df['expiry_date'].dtype)

Before:
list_date dtype:   object
expiry_date dtype: object
Sample list_date:  ['2021-01-28', '2022-11-29', '2022-07-29']
Sample expiry_date: ['2021-06-08', '2023-03-02', '2023-01-11']

After:
list_date dtype:   datetime64[ns]
expiry_date dtype: datetime64[ns]


In [62]:
# Step 3: Standardize listing_status Column

print("Before - unique values:", df['listing_status'].unique())

df['listing_status'] = df['listing_status'].str.strip().str.title()

print("After  - unique values:", df['listing_status'].unique())

Before - unique values: ['Sold' 'Expired' 'Active' 'Withdrawn' 'Pending']
After  - unique values: ['Sold' 'Expired' 'Active' 'Withdrawn' 'Pending']


In [63]:
df[df['days_on_market'].isnull()]['listing_status']  .value_counts()

listing_status
Sold         302
Active       124
Expired       79
Withdrawn     59
Pending       27
Name: count, dtype: int64

### Step 4: Handle Missing Values in `days_on_market`
**Approach: Date-Based Imputation**
Null values in `days_on_market` were filled using existing columns in the dataset:
`expiry_date - list_date = actual days on market`

In [64]:
# Step 4: Handle Missing Values in days_on_market

print("Before - Nulls:", df['days_on_market'].isnull().sum())

null_mask = df['days_on_market'].isnull()
df.loc[null_mask, 'days_on_market'] = (
    df.loc[null_mask, 'expiry_date'] - df.loc[null_mask, 'list_date']
).dt.days

print("After  - Nulls:", df['days_on_market'].isnull().sum())
print("\ndays_on_market stats by status:")
print(df.groupby('listing_status')['days_on_market'].median().round(1))

Before - Nulls: 591
After  - Nulls: 0

days_on_market stats by status:
listing_status
Active       166.5
Expired      163.0
Pending      176.5
Sold         170.0
Withdrawn    165.5
Name: days_on_market, dtype: float64


### Step 5: Check Outliers in `list_price`
**Approach: IQR Method**
Using IQR (Interquartile Range) to calculate lower and upper bounds — values outside this range are considered outliers.

In [65]:
# Step 5: Check Outliers in list_price

print("Basic Stats:")
print(df['list_price'].describe().apply(lambda x: f"{x:,.0f}"))

Q1 = df['list_price'].quantile(0.25)
Q3 = df['list_price'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['list_price'] < lower_bound) | (df['list_price'] > upper_bound)]

print(f"\nQ1: {Q1:,.0f}  |  Q3: {Q3:,.0f}  |  IQR: {IQR:,.0f}")
print(f"Lower Bound: {lower_bound:,.0f}")
print(f"Upper Bound: {upper_bound:,.0f}")
print(f"\nOutliers found: {len(outliers)} rows")
print(outliers['list_price'].describe().apply(lambda x: f"{x:,.0f}"))

Basic Stats:
count        6,000
mean     1,406,376
std        775,285
min         51,000
25%        710,450
50%      1,407,500
75%      2,051,200
max      2,999,100
Name: list_price, dtype: object

Q1: 710,450  |  Q3: 2,051,200  |  IQR: 1,340,750
Lower Bound: -1,300,675
Upper Bound: 4,062,325

Outliers found: 0 rows
count      0
mean     nan
std      nan
min      nan
25%      nan
50%      nan
75%      nan
max      nan
Name: list_price, dtype: object


### Step 6: Convert `days_on_market` float → int
Days should not have decimal values, so the dtype is converted to integer.

In [66]:
# Step 6: Convert days_on_market to int

print("Before dtype:", df['days_on_market'].dtype)
print("Sample:", df['days_on_market'].head(3).tolist())

df['days_on_market'] = df['days_on_market'].astype(int)

print("\nAfter dtype: ", df['days_on_market'].dtype)
print("Sample:", df['days_on_market'].head(3).tolist())

Before dtype: float64
Sample: [110.0, 145.0, 349.0]

After dtype:  int64
Sample: [110, 145, 349]


### Step 7: Clean `description` Column
Remove extra leading/trailing whitespace from description text.

In [67]:
# Step 7: Clean description column

print("Before sample:")
print(df['description'].head(3).tolist())

df['description'] = df['description'].str.strip()

print("\nAfter sample:")
print(df['description'].head(3).tolist())

Before sample:
['Building major hold buy western low chair.', 'Hope gas difficult think southern.', 'Much security purpose return image indeed nation raise. Instead imagine clear seven check.']

After sample:
['Building major hold buy western low chair.', 'Hope gas difficult think southern.', 'Much security purpose return image indeed nation raise. Instead imagine clear seven check.']
